# Self-excited instability of a Rijke tube

A **Rijke tube** is the textbook thermoacoustic oscillator: a duct with a heat source that, for the right time lag, features a self-excited thermoacoustic instability.
It is the simplest system in which an *active* element (a heat-releasing element that responds to the acoustic field) pumps energy into a duct mode and drives it unstable.

The mechanism is **Rayleigh's criterion**: when the unsteady heat release $q'$ is in phase with the pressure fluctuation $p'$ at the flame, the cycle gains energy and the oscillation grows.
We model the flame response with a **flame transfer function** $\mathcal{F}(\omega)$ relating unsteady heat release to upstream velocity fluctuations:

$$
\frac{\dot{Q}'(\omega)}{\overline{ \dot{Q}}} \;=\; \mathcal{F}(\omega)\,\frac{u'_{\mathrm{ref}}(\omega)}{\overline u_{\mathrm{ref}}}.
$$

Here we take the classic $n$-$\tau$ model,

$$
\mathcal{F}(\omega) = n\,e^{-i\omega\tau},
$$

so the heat release responds to the velocity an instant $\tau$ earlier, with a constant gain $n$.
The reference position is the acoustic "state" immediately before the flame.

In FNS this is modeled in the  **source matrix** $\mathbf S(\omega)$ of the perturbation operator $\mathbf A(\omega) = \mathbf J_{\mathrm{alg}} + i\omega\mathbf M + \mathbf P(\omega) + \mathbf S(\omega)$ (theory.md §12.4).
The mean flame is acoustically passive (does not respond to acoustic fluctuations); attaching a `DynamicSource` makes $\mathbf A$ non-self-adjoint, and the **stability eigenproblem** $\det\mathbf A(\omega)=0$ acquires complex roots: modal frequency $\mathrm{Re}(\omega)/2\pi$ and growth rate $-\mathrm{Im}(\omega)$.

In this notebook, we will (1) build the mean flow, (2) attach an $n$-$\tau$ flame, (3) find the unstable mode, (4) map the stability boundary against the **analytical** compact-flame dispersion relation, and (5) repeat with the **equilibrium flame** model driven by the same FTF.

In [ ]:
import os, sys, warnings

_root = os.getcwd()
while not os.path.isdir(os.path.join(_root, "fns")) and _root != os.path.dirname(_root):
    _root = os.path.dirname(_root)
sys.path.insert(0, _root)

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from fns.elements import catalog as cat
from fns.elements.dynamic_source import n_tau, n_tau_flame
from fns.perturbation.boundary_bc import PerturbationBC
from fns.perturbation import eigenmodes, forced_response
from fns.perturbation.terminals import find_terminals
from fns.perturbation.power import boundary_power, acoustic_intensity
from fns.plotting import use_fns_theme, plot_transfer_function
from fns.shell import Network
from fns.derive import ES_RHO, ES_C, ES_U, ES_P, ES_T, ES_M, ES_AREA
from fns.thermo.configure import perfect_gas

# Below is because plotly LaTeX rendering is not working out of box in notebooks viewed in VSCode/Cursor.
from IPython.display import display, HTML
import plotly.offline as pyo
pyo.init_notebook_mode()
display(HTML(
    '<script src="https://cdnjs.cloudflare.com/ajax/libs/mathjax/2.7.7/MathJax.js?config=TeX-AMS-MML_SVG"></script>'
))

_ = use_fns_theme()

## 1. The mean flow

Cold air enters a rigid ($\dot{m}'=0$) mass flow inlet, flows down through a duct of length $L_1$, crosses an acoustically compact flame, and leaves through a duct of length $L_2$ to an open end ($p'=0$).
We keep the mean Mach number low so the acoustics are clean.

The flame is a **perfect-gas heat-release flame** raising the total temperature by $\Delta T = \dot Q/(\dot m c_p)$.
A `DynamicSource` ($n$-$\tau$ flame transfer function) is attached to it; the steady solve ignores it (a constant mean source is acoustically passive), so the mean flow is the same with or without it.

In [ ]:
# Constants that define the perfect gas model
R, GAMMA = 287.0, 1.4
CP = GAMMA * R / (GAMMA - 1.0)
# Tube cross-section
AREA = 0.01
# Cold (upstream) and hot (downstream) duct lengths
L1, L2 = 0.6, 0.4
# Mean mass flow rate through the duct
MDOT = 0.006
# Cold inlet temperature
T_IN = 300.0
# Open-end (and reference) static pressure
P_OUT = 1.0e5
# Mean temperature rise across the flame, used here to size the flame heat release rate.
DT_RISE = 400.0
# Integrated heat release rate this compact flame will produce [W]
QDOT = MDOT * CP * DT_RISE


def rijke(
    n,
    tau,
    *,
    R=R,
    gamma=GAMMA,
    area=AREA,
    L_cold=L1,
    L_hot=L2,
    mdot=MDOT,
    qdot=QDOT,
    T_inlet=T_IN,
    p_outlet=P_OUT,
    drive=False,
):
    """Build and solve the Rijke tube with a compact n-tau flame.

    Every geometric, mean-flow, and gas parameter is an explicit argument (defaulting to the
    module-level constants), and the network is assembled through the ``Network`` object so the
    inlet -> duct -> flame -> duct -> outlet topology is spelled out node by node.  The flame's
    velocity reference is the edge just upstream of it; rather than guess that index up front, we
    take it from the value ``connect`` returns and attach the n-tau source afterwards with
    ``set_dynamic_source``.

    Parameters
    ----------
    n : float
        Flame-transfer-function gain of the n-tau source.
    tau : float
        Flame time lag [s] of the n-tau source.
    R : float
        Specific gas constant [J/(kg K)] of the perfect gas.
    gamma : float
        Heat-capacity ratio of the perfect gas.
    area : float
        Tube cross-sectional area [m^2] (uniform along the duct).
    L_cold : float
        Cold (upstream) duct length [m].
    L_hot : float
        Hot (downstream) duct length [m].
    mdot : float
        Mean mass flow rate through the duct [kg/s].
    qdot : float
        Integrated mean heat release rate of the compact flame [W].
    T_inlet : float
        Cold inlet temperature [K].
    p_outlet : float
        Open-end static pressure [Pa]; also the reference pressure.
    drive : bool
        When ``True`` the open end is additionally driven by a unit incoming acoustic wave;
        the convective open-end reflection -(1-M)/(1+M) is kept, so the tube's modes are unchanged and only
        a forcing is added -- this is what the forced-response frequency sweep excites.  The mean
        flow is identical to ``drive=False``.  Default ``False``.

    Returns
    -------
    Solution
        The converged mean-flow solution (carrying ``.problem``, ``.x``, ``.table()``, and
        ``.print_states()``).
    """
    # Create the network object
    net = Network(perfect_gas(R, gamma), p_ref=p_outlet, T_ref=T_inlet, mdot_ref=mdot)

    # Add the elements (the flame's dynamic source is attached later, once its reference edge exists).
    # When `drive` is set, the open end injects a unit incoming wave on top of its convective-neutral reflection R = -(1-M)/(1+M),
    # so the sweep can excite the tube without altering its modes.
    outlet_bc = PerturbationBC.mean_flow_open_end(driven=("acoustic",) if drive else ())
    idx_inlet = net.add(cat.mass_flow_inlet(mdot, T_inlet))
    idx_cold = net.add(cat.duct(L_cold))
    idx_flame = net.add(cat.heat_release_flame(qdot))
    idx_hot = net.add(cat.duct(L_hot))
    idx_outlet = net.add(cat.pressure_outlet(p_outlet, perturbation_bc=outlet_bc))

    # Assemble the network: cold air -> duct -> n-tau heat-release flame -> duct -> open end.
    # connect() returns the new edge's id; the cold -> flame edge is the flame's velocity reference.
    net.connect(idx_inlet, idx_cold, area)
    idx_ref = net.connect(idx_cold, idx_flame, area)
    net.connect(idx_flame, idx_hot, area)
    net.connect(idx_hot, idx_outlet, area)

    # Attach the n-tau flame transfer function now that the reference edge id is known
    net.set_dynamic_source(idx_flame, n_tau_flame(n, tau, ref_edge=idx_ref))

    # Solve the mean flow and ensure convergence
    sol = net.solve()
    assert sol.converged

    return sol


sol0 = rijke(0.0, 0.0)
# edge 1 is just upstream of the flame; edge 2 the hot products just downstream
sol0.print_states()

## 2. The flame transfer function

The $n$-$\tau$ model is a pure gain $n$ and a delay $\tau$: a flat magnitude and a phase that winds linearly with frequency.
The `plot_transfer_function` helper offers tools to plot a complex valued transfer function.

In [ ]:
N, TAU = 0.8, 4.0e-3       # gain and time lag [s]
ftf = n_tau(N, TAU)
freqs = np.linspace(0.0, 400.0, 400)

plot_transfer_function(
    ftf, freqs, names=[f"n-tau  (n={N}, tau={TAU*1e3:.1f} ms)"],
    title=r"Flame transfer function",
).show()

## 3. The unstable mode

A self-excited mode is a complex root of $\det\mathbf A(\omega)=0$.
Before hunting those roots directly, we take the **experimentalist's route**: drive one end of the tube and sweep a *real* frequency, watching two things -- the acoustic energy stored inside the duct, and the net acoustic power crossing the boundaries.
Resonances show up as energy peaks; the boundary-power trace shows whether the domain is absorbing the drive or feeding energy back out.
This locates the modes (and already reveals the flame's effect) without any contour machinery.

We then **pin** each mode with `eigenmodes`, which solves $\det\mathbf A(\omega)=0$ by Beyn's contour-integral method, returning the exact complex frequency and growth rate.
For this example we use the isentropic (no convected entropy) operator -- the standard acoustic network model -- so the flame's energy jump stays physical while entropy convection is dropped.
The boundaries are energy-neutral, so a **positive growth rate** is a genuinely self-excited (unstable) mode.

### A forced frequency sweep

We drive the open end with a unit incoming acoustic wave (keeping its neutral convective open-end reflection $R=-(1-M)/(1+M)$, so the tube's own modes are untouched -- we merely *excite* them) and solve the forced perturbation field at each real frequency with `forced_response`.
Two diagnostics are read off the field:

- **Energy in the duct.**  The network stores the wave amplitudes only at the edge stations, so inside each uniform duct we reconstruct the field analytically ($f$ rides downstream at $\overline u+\overline c$, $g$ upstream at $\overline c-\overline u$) and integrate the Myers energy density $e=\tfrac12\overline\rho\,[(1+M)|f|^2+(1-M)|g|^2]$ along its length.
- **Power across the boundaries.**  The net Myers acoustic intensity $I=\tfrac12\overline\rho\,\overline c\,[(1+M)^2|f|^2-(1-M)^2|g|^2]$ through each terminal face, summed and signed *into* the domain.

We sweep the passive ($n=0$) and active ($n=0.8$, $\tau=4$ ms) tube together; each curve is normalised to its own peak so both are visible (the passive tube responds far more strongly to the same drive, its modes being only lightly damped).

In [ ]:
# The cold duct spans edges (0, 1) and the hot duct edges (2, 3): the tail (upstream) and head
# (downstream) station of each, in the build order of `rijke`.
DUCTS = [(0, 1, L1), (2, 3, L2)]
_trapz = np.trapezoid if hasattr(np, "trapezoid") else np.trapz


def domain_energy(fr, n_samp=120):
    """Acoustic energy stored in the duct volumes at each swept frequency.

    The field is known only at the edge stations, so inside each uniform duct it is reconstructed
    analytically -- f rides downstream at u+c, g upstream at c-u -- and the Myers energy density
    (cf. fns.perturbation.power.acoustic_energy_density) is integrated along the duct.
    """
    est = fr.est
    omega = 2.0 * np.pi * fr.freqs
    s = np.linspace(0.0, 1.0, n_samp)
    E = np.zeros(fr.freqs.size)
    for tail, head, L in DUCTS:
        rho, c, u = float(est[ES_RHO, tail]), float(est[ES_C, tail]), float(est[ES_U, tail])
        M, area = float(est[ES_M, tail]), float(est[ES_AREA, tail])
        x = s * L
        f = fr.waves(tail)[:, 0][:, None] * np.exp(-1j * omega[:, None] * x[None, :] / (u + c))        # downstream wave
        g = fr.waves(head)[:, 1][:, None] * np.exp(-1j * omega[:, None] * (L - x)[None, :] / (c - u))   # upstream wave
        e = 0.5 * rho * ((1.0 + M) * np.abs(f) ** 2 + (1.0 - M) * np.abs(g) ** 2)
        E += _trapz(e, x, axis=1) * area
    return E


def net_boundary_power(fr, prob):
    """Net acoustic power crossing all boundaries *into* the domain, at each swept frequency."""
    est = fr.est
    P = np.zeros(fr.freqs.size)
    for t in find_terminals(prob):
        e = t.edge
        rho, c, M, area = float(est[ES_RHO, e]), float(est[ES_C, e]), float(est[ES_M, e]), float(est[ES_AREA, e])
        w = fr.waves(e)
        flux = np.array([acoustic_intensity(rho, c, M, w[j, 0], w[j, 1]) for j in range(fr.freqs.size)]) * area
        P += flux if t.at_tail else -flux        # sign so positive is power flowing into the domain
    return P


freqs_sweep = np.linspace(60.0, 320.0, 600)
fig = make_subplots(
    rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.09,
    subplot_titles=("Acoustic energy stored in the duct", "Net acoustic power crossing the boundaries"),
)
colors = {"passive": "#1f77b4", "active": "#d62728"}
for label, n, tau in [("passive", 0.0, 0.0), ("active", N, TAU)]:
    sol_d = rijke(n, tau, drive=True)
    fr = forced_response(sol_d.problem, sol_d.x, freqs_sweep, isentropic=True)
    E = domain_energy(fr)
    P = net_boundary_power(fr, sol_d.problem)
    # the fundamental resonance (largest energy peak below 200 Hz)
    f_fund = freqs_sweep[np.argmax(np.where(freqs_sweep < 200.0, E, 0.0))]
    fig.add_scatter(x=freqs_sweep, y=E / E.max(), name=label, legendgroup=label,
                    line=dict(color=colors[label]), row=1, col=1)
    fig.add_scatter(x=freqs_sweep, y=P / np.abs(P).max(), name=label, legendgroup=label, showlegend=False,
                    line=dict(color=colors[label]), row=2, col=1)
    fig.add_vline(x=f_fund, line_dash="dot", line_color=colors[label], row=1, col=1,
                  annotation_text=f"{label}  {f_fund:.0f} Hz", annotation_position="top")
fig.add_hline(y=0.0, line_width=1, line_color="#888", row=2, col=1)
fig.update_yaxes(title_text="energy  (norm. to peak)", row=1, col=1)
fig.update_yaxes(title_text="power in  (norm. to peak)", row=2, col=1)
fig.update_xaxes(title_text="frequency  (Hz)", row=2, col=1)
fig.update_layout(title="Forced frequency sweep: drive the open end, watch the response", template="fns")
fig.show()

The energy peaks are the tube's resonances -- our mode candidates.
With the flame inert they sit at the bare duct frequencies ($\approx 111$ and $\approx 306$ Hz); **switching the flame on slides the fundamental up to $\approx 136$ Hz** and broadens it.
(The active resonance is broad and weak in this real-frequency view because a strongly growing mode has its pole well off the real axis, so a real-frequency drive only grazes it.)

The boundary-power trace tells the energy story.
The passive tube only ever **absorbs** the drive ($P>0$) -- a passive resonator dissipating the injected power.
The active tube instead **emits** net acoustic power ($P<0$) near its unstable fundamental: the flame produces more acoustic energy than the boundaries and dissipation carry off, and the excess radiates out.
That sign flip is the experimental fingerprint of self-excitation.

We now pin the modes exactly.  **Passive baseline first** ($n=0$):

In [ ]:
# Passive baseline: with the flame inert (n=0) the network is acoustically passive, so these are the
# bare duct modes -- the frequencies the active flame will act on.
# The growth rates are near-neutral; the small negative value is the O(M) mean-flow damping the
# energy-neutral ends leave behind.
sol_p = rijke(0.0, 0.0)
res_p = eigenmodes(sol_p.problem, sol_p.x, freq_band=(40.0, 320.0), growth_band=(-200.0, 200.0), isentropic=True)

for m in res_p.summary():
    print(f"f = {m['freq_hz']:7.2f} Hz    growth = {m['growth_rate']:+8.2f} 1/s")

res_p.plot_spectrum(title="Rijke spectrum -- passive flame (n=0)").show()

### Switching the flame on

Turning the flame on adds the source $\mathbf S(\omega)=\overline{Q}\,\mathcal F(\omega)$ to the operator, with $\mathcal F(\omega)=n\,e^{-i\omega\tau}$.
This is a **complex, frequency-dependent** term, so it does not simply add growth at the passive frequency -- it moves the eigenvalue in the complex plane:

- the **imaginary** part ($\propto -\sin\omega\tau$) is resistive and feeds or drains the **growth rate**;
- the **real** part ($\propto \cos\omega\tau$) is reactive and **pulls the modal frequency**.

So the fundamental does not sit at the passive $\approx 111\,\mathrm{Hz}$ and merely cross into instability -- it shifts up to $\approx 136\,\mathrm{Hz}$ as it goes unstable.
This frequency pull is genuine thermoacoustics, not a numerical artefact: the analytic dispersion relation in §4 reproduces the same shifted, growing root at every $\tau$ (and at $n=0$ collapses back onto the passive baseline above).

In [ ]:
sol = rijke(N, TAU)
res = eigenmodes(sol.problem, sol.x, freq_band=(40.0, 320.0), growth_band=(-200.0, 200.0), isentropic=True)

for m in res.summary():
    flag = "  <-- UNSTABLE" if m["unstable"] else ""
    print(f"f = {m['freq_hz']:7.2f} Hz    growth = {m['growth_rate']:+8.2f} 1/s{flag}")

res.plot_spectrum(title=f"Rijke spectrum  (n={N}, tau={TAU*1e3:.0f} ms)").show()

We can **show** the energy-neutral claim directly with the acoustic-power diagnostic.
`boundary_power` reads each terminal's reflection magnitude $|R|$ against the energy-neutral bound $(1\pm M)/(1\mp M)$ and the net acoustic power it passes into the domain.
For the unstable mode that net is $\approx 0$: the rigid inlet ($u'=0$) and the open outlet ($p'=0$) are acoustic nodes, so no acoustic energy crosses them -- the growth is supplied by the flame source alone.

In [ ]:
# Show the boundaries are energy-neutral with the acoustic-power diagnostic: for the unstable mode the
# net acoustic power crossing the terminals is ~0, so the growth is fed by the flame source, not the
# ends (each terminal is an acoustic node -- the mass-flow inlet pins u'=0, the open outlet pins p'=0 --
# so the Myers energy flux vanishes at the face).
imode = int(np.argmax(res.growth_rates))   # the self-excited (unstable) mode
bp = boundary_power(res, mode=imode)

print(f"unstable mode  f = {bp.freq_hz:.2f} Hz   growth = {bp.growth_rate:+.2f} 1/s\n")
print(f"{'terminal':<10}{'|R|':>8}{'energy-neutral |R|':>22}{'power in':>14}")
for e in sorted(bp.entries, key=lambda d: d["edge"]):
    print(f"{e['name']:<10}{e['reflection']:>8.3f}{e['passive_bound']:>22.3f}{e['power_in']:>14.1e}")
print(f"\nnet boundary acoustic power = {bp.net:+.1e}  (mode-scale units): ~0, the boundaries are")
print("energy-neutral -- the +growth is supplied internally by the flame, not the terminals.")

### The shape of the unstable mode

The growth rate and frequency say *when* and *how fast*; the **mode shape** says *where*.
Reconstructing the pressure field analytically inside each duct and animating it over one cycle
shows the self-excited standing wave, with the compact flame marked at the cold/hot interface
($x=L_1$).
The reconstruction is exact — the duct's own phase relation evaluated at every interior station,
not a discretization.

In [ ]:
# Spatial shape of the self-excited mode: the pressure field reconstructed analytically inside each
# duct (f rides downstream at u+c, g upstream at c-u) and animated over a cycle.  The flame sits at
# the cold/hot interface (x = L1); press play to watch the growing standing wave.
res.animate_mode(
    imode, variable="p",
    title=f"Unstable Rijke mode: p' along the tube ({res.freqs[imode]:.0f} Hz)",
).show()

## 4. The stability map -- and analytical verification

Sweeping the time lag $\tau$ traces the classic $n$--$\tau$ stability band: the peak growth rate rises and falls, crossing zero into instability for a range of $\tau$.

We check FNS against the **analytical compact-flame dispersion relation** -- a two-region (cold/hot) acoustic tube with the zero-mean-Mach jump

$$ p'_1 = p'_2, \qquad u'_2 - u'_1 = \frac{\gamma-1}{\gamma\,\overline{p}\,A}\,q', $$

and $q' = \overline{Q}\,\mathcal{F}(\omega)\,u'_1/\overline{u}_1$ with $\mathcal{F}(\omega)=n\,e^{-i\omega\tau}$.

**Modal ansatz.**  Seek normal modes $\hat\xi(x,t)=\Re\{\hat\xi(x)\,e^{\lambda t}\}$.
FNS uses the harmonic convention $e^{i\omega t}$, so each eigenvalue is

$$ \lambda = i\omega, \qquad \omega = \omega_r + i\omega_i \in \mathbb{C}. $$

The real and imaginary parts of $\lambda$ are the quantities plotted in the spectrum:

- $\mathrm{Re}\,\lambda = -\mathrm{Im}\,\omega$ -- exponential growth rate [1/s] ($\mathrm{Re}\,\lambda>0$ is unstable);
- $\mathrm{Im}\,\lambda = \mathrm{Re}\,\omega$ -- angular frequency [rad/s], with $f=\mathrm{Im}\,\lambda/(2\pi)$.

In each uniform region $j\in\{1,2\}$ (cold / hot) the spatial part is a pair of plane waves, $k_j=\omega/c_j$, $Z_j=\rho_j c_j$:

$$ \hat p'_j(x) = Z_j\big(A_j^+ e^{ik_j x} + A_j^- e^{-ik_j x}\big), $$

with the flame at $x=0$.
Matching the inlet hard wall ($\hat u'_1=0$ at $x=-L_1$), the open outlet ($\hat p'_2=0$ at $x=L_2$), pressure continuity, and the velocity jump above gives a $4\times4$ system $\mathbf M(\omega)\mathbf a=0$ for the wave amplitudes $\mathbf a$.
A nontrivial mode exists iff $\det\mathbf M(\omega)=0$ (equivalently $\det\mathbf A(\omega)=0$ in FNS).
Each root $\omega_k$ yields an eigenvalue $\lambda_k=i\omega_k$; there is no closed form for $\lambda_k$ in general, but Newton polish on $\det\mathbf M$ recovers it.

At this low Mach the analytic and FNS roots agree closely; the small residual is the $O(M)$ mean-flow damping the zero-Mach analytic omits.

In [ ]:
def region_means(sol):
    est = sol.table()
    return dict(rho1=est[ES_RHO, 1], c1=est[ES_C, 1], u1=est[ES_U, 1], p1=est[ES_P, 1],
                rho2=est[ES_RHO, 2], c2=est[ES_C, 2])

def analytic_det(omega, m, Qdot, n, tau):
    k1, k2 = omega / m['c1'], omega / m['c2']
    Z1, Z2 = m['rho1'] * m['c1'], m['rho2'] * m['c2']
    eP1, eM1 = np.exp(1j * k1 * L1), np.exp(-1j * k1 * L1)
    eP2, eM2 = np.exp(-1j * k2 * L2), np.exp(1j * k2 * L2)
    Th = (GAMMA - 1.0) * Qdot / (GAMMA * m['p1'] * AREA) * (n * np.exp(-1j * omega * tau)) / (Z1 * m['u1'])
    M = np.array([[eP1, -eM1, 0, 0],
                  [1, 1, -1, -1],
                  [-(1 / Z1 + Th), (1 / Z1 + Th), 1 / Z2, -1 / Z2],
                  [0, 0, eP2, eM2]], dtype=complex)
    return np.linalg.det(M)

def analytic_modes(m, Qdot, n, tau, fband=(40.0, 320.0)):
    wr = 2 * np.pi * np.linspace(*fband, 60)
    wi = np.linspace(-220.0, 220.0, 60)
    WR, WI = np.meshgrid(wr, wi)
    D = np.abs(np.vectorize(lambda w: analytic_det(w, m, Qdot, n, tau))(WR + 1j * WI))
    grid = WR + 1j * WI
    seeds = [grid[i, j] for i in range(1, D.shape[0] - 1) for j in range(1, D.shape[1] - 1)
             if D[i, j] == D[i - 1:i + 2, j - 1:j + 2].min()]
    out = []
    for w0 in seeds:
        w = complex(w0)
        for _ in range(60):
            h = 1e-3 * (abs(w) + 1.0)
            d = analytic_det(w, m, Qdot, n, tau)
            dp = (analytic_det(w + h, m, Qdot, n, tau) - analytic_det(w - h, m, Qdot, n, tau)) / (2 * h)
            if dp == 0:
                break
            s = d / dp
            w -= s
            if abs(s) < 1e-9:
                break
        if 2 * np.pi * fband[0] < w.real < 2 * np.pi * fband[1] and -220.0 < w.imag < 220.0:
            if all(abs(w - o) > 1e-1 for o in out):
                out.append(w)
    return out

m_mean = region_means(rijke(0.0, 0.0))
taus = np.linspace(0.0, 6.0e-3, 25)
g_fns, g_an = [], []
for tau in taus:
    sol = rijke(N, tau)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        r = eigenmodes(sol.problem, sol.x, freq_band=(40.0, 320.0), growth_band=(-260.0, 260.0), isentropic=True)
    g_fns.append(max(r.growth_rates) if len(r) else np.nan)
    am = analytic_modes(m_mean, QDOT, N, tau)
    g_an.append(max((-w.imag for w in am), default=np.nan))

fig = go.Figure()
fig.add_scatter(x=taus * 1e3, y=g_an, mode="lines", name="analytic")
fig.add_scatter(x=taus * 1e3, y=g_fns, mode="markers", name="FNS eigenmodes", marker=dict(size=8))
fig.add_hline(y=0.0, line_dash="dash", annotation_text="stability boundary")
fig.update_layout(title="Rijke stability map: peak growth rate vs. flame time lag",
                  xaxis_title="time lag  tau  (ms)", yaxis_title="growth rate  (1/s)", template="fns")
fig.show()

The FNS markers sit on the analytic curve and cross the stability boundary at the same lags -- the sign **and** magnitude of the heat-release coupling check out: the dynamic source genuinely drives the instability, not merely perturbs it.

## 5. The reacting flame

The same FTF drives the **reacting equilibrium flame** -- the source is flame-agnostic.
Here the mean flame temperature is not prescribed: stoichiometric H$_2$/air enters frozen and burns to chemical equilibrium across the flame (the reacting mean solver), and the mean heat release $\overline{Q}$ for the FTF auto-derives from the converged temperature rise.

The reacting acoustics use the gas's **actual caloric coupling** in the characteristic maps -- the per-edge $(\partial h/\partial\rho)_p,\ (\partial h/\partial p)_\rho$ from the equilibrium closure, *not* the perfect-gas $c_p/R$ -- so the reacting case is now **quantitatively** verified, not just demonstrated.
Two physical differences from the fixed-power perfect-gas flame are worth noting:

* The equilibrium flame **conserves total enthalpy** (the heat release is internal, in the absolute-enthalpy datum), so its compact jump is *mass-flux continuous* ($\rho u'$ continuous), not velocity-continuous.
* That gives the flame an **intrinsic quasi-steady response** (its heat release tracks the reactant supply), so the effective FTF is $1 + n\,e^{-i\omega\tau}$ -- it takes a larger lag to destabilize than the perfect-gas flame.

We verify against the matching **equilibrium-flame dispersion relation**: five waves (the two acoustic + an entropy spot shed at the flame) with the exact equilibrium EOS derivatives and the same source coupling FNS stamps.

In [ ]:
from thermolib import ThermoInp, Thermo
from fns.composition import resolve_composition, enthalpy_mass
from fns.thermo.configure import equilibrium
from fns.thermo.api import EQ_FROZEN, EQ_KERNEL

THERMO_INP = os.path.join(_root, "thermolib", "data", "thermo.inp")
lib = ThermoInp(THERMO_INP).library(["H2", "O2", "N2", "H2O", "OH", "H", "O", "HO2", "NO"])
gas = Thermo(lib)
fuel_air = {"H2": 1.0, "O2": 0.5, "N2": 0.5 * 3.76}   # stoichiometric, mole basis
Y, Z = resolve_composition(lib, fuel_air, basis="mole")
H_REACT = enthalpy_mass(lib, Y, 300.0)
MDOT_R = 0.02


def rijke_reacting(
    n,
    tau,
    *,
    area=AREA,
    L_cold=L1,
    L_hot=L2,
    mdot=MDOT_R,
    T_inlet=300.0,
    p_outlet=1.0e5,
    composition=fuel_air,
    h_inlet=H_REACT,
):
    """Build and solve the reacting (equilibrium-flame) Rijke tube.

    Same explicit-parameter, ``Network``-built topology as ``rijke``, but the flame temperature
    is not prescribed: the reactant stream enters frozen (unburnt) and burns to chemical
    equilibrium across the flame.  The cold approach edges carry the frozen closure and the hot
    product edges the equilibrium closure, set per edge via ``connect(..., edge_model=...)``.  As
    in ``rijke``, the flame's velocity reference is taken from the edge id ``connect`` returns and
    the n-tau source is attached afterwards with ``set_dynamic_source``.

    Parameters
    ----------
    n : float
        Flame-transfer-function gain of the n-tau source.
    tau : float
        Flame time lag [s] of the n-tau source.
    area : float
        Tube cross-sectional area [m^2] (uniform along the duct).
    L_cold : float
        Cold (upstream) duct length [m].
    L_hot : float
        Hot (downstream) duct length [m].
    mdot : float
        Mean mass flow rate through the duct [kg/s].
    T_inlet : float
        Reactant inlet temperature [K].
    p_outlet : float
        Open-end static pressure [Pa]; also the reference pressure.
    composition : dict
        Inlet (and backflow) reactant composition.
    h_inlet : float
        Absolute-enthalpy datum of the reactant stream [J/kg].

    Returns
    -------
    Solution
        The converged mean-flow solution (carrying ``.problem``, ``.x``, ``.table()``, and
        ``.print_states()``).
    """
    # unburnt reactant approach -> burnt products: frozen upstream, equilibrium downstream of the flame
    net = Network(equilibrium(gas.mech), p_ref=p_outlet, T_ref=T_inlet, mdot_ref=mdot, h_ref=h_inlet)
    inlet = net.add(
        cat.mass_flow_inlet(
            mdot, T_inlet, composition=composition, basis="mole", perturbation_bc=PerturbationBC.hard_wall()
        )
    )
    cold = net.add(cat.duct(L_cold))
    flame = net.add(cat.equilibrium_flame())
    hot = net.add(cat.duct(L_hot))
    outlet = net.add(
        cat.pressure_outlet(
            p_outlet, Tt_backflow=T_inlet, composition=composition, basis="mole",
            perturbation_bc=PerturbationBC.open_end(),
        )
    )
    # connect() returns the new edge's id; the cold -> flame edge is the flame's velocity reference
    net.connect(inlet, cold, area, edge_model=EQ_FROZEN)
    ref = net.connect(cold, flame, area, edge_model=EQ_FROZEN)
    net.connect(flame, hot, area, edge_model=EQ_KERNEL)
    net.connect(hot, outlet, area, edge_model=EQ_KERNEL)
    # attach the n-tau flame transfer function now that the reference edge id is known
    net.set_dynamic_source(flame, n_tau_flame(n, tau, ref_edge=ref))
    sol = net.solve()
    assert sol.converged
    return sol


sol_r = rijke_reacting(0.0, 0.0)
# edge 1 is the unburnt reactant approach, edge 2 the burnt products across the (adiabatic) flame
sol_r.print_states()

In [ ]:
from fns.thermo.api import thermo_state


def reacting_caloric(prob, x, est, e):
    # (dh/drho)_p, (dh/dp)_rho at edge e via a complex step of the equilibrium closure
    xi = np.ascontiguousarray(x[3:3 + prob.n_elem, e]).astype(np.complex128)
    ht, p, d = complex(x[2, e]), float(est[ES_P, e]), 1e-30
    mid = int(prob.edge_model[e])
    drho_dh = thermo_state(mid, prob.tf, prob.ti, xi, ht + 1j * d, complex(p))[1].imag / d
    drho_dp = thermo_state(mid, prob.tf, prob.ti, xi, ht, p + 1j * d)[1].imag / d
    return 1.0 / drho_dh, -drho_dp / drho_dh


def reacting_det(omega, mm, cal1, cal2, delta, n, tau):
    # 5 waves [f1, g1, f2, g2, h2]: mass-flux + momentum + (h_t cont + source), equilibrium EOS
    rho1, cc1, u1, p1, rho2, cc2, u2, p2 = mm
    a1, b1 = cal1
    a2, b2 = cal2
    Z1, Z2 = rho1 * cc1, rho2 * cc2
    F = n * np.exp(-1j * omega * tau)
    k1p, k1m, k2p, k2m = omega / (u1 + cc1), omega / (cc1 - u1), omega / (u2 + cc2), omega / (cc2 - u2)
    M = np.zeros((5, 5), complex)
    M[0, 0], M[0, 1] = np.exp(1j * k1p * L1), -np.exp(-1j * k1m * L1)        # inlet hard wall
    M[1, 2], M[1, 3] = np.exp(-1j * k2p * L2), np.exp(1j * k2m * L2)         # outlet open end
    M[2] = [AREA * (u1 * Z1 / cc1**2 + rho1), AREA * (u1 * Z1 / cc1**2 - rho1),
            -AREA * (u2 * Z2 / cc2**2 + rho2), -AREA * (u2 * Z2 / cc2**2 - rho2), -AREA * u2]   # mass flux
    # momentum
    M[3] = [Z1 + u1**2 * Z1 / cc1**2 + 2 * rho1 * u1, Z1 + u1**2 * Z1 / cc1**2 - 2 * rho1 * u1,
            -(Z2 + u2**2 * Z2 / cc2**2 + 2 * rho2 * u2), -(Z2 + u2**2 * Z2 / cc2**2 - 2 * rho2 * u2), -u2**2]
    ht1, ht2, src = a1 * Z1 / cc1**2 + b1 * Z1, a2 * Z2 / cc2**2 + b2 * Z2, delta * F / u1
    M[4] = [-ht1 - src, -ht1 + src, ht2, ht2, a2]                            # energy + heat-release source
    return np.linalg.det(M)


def reacting_mode(mm, cal1, cal2, delta, n, tau, seed):
    w = complex(seed)
    for _ in range(200):
        h = 1e-3 * (abs(w) + 1.0)
        dp = (
            reacting_det(w + h, mm, cal1, cal2, delta, n, tau) - reacting_det(w - h, mm, cal1, cal2, delta, n, tau)
        ) / (2 * h)
        if dp == 0:
            break
        s = reacting_det(w, mm, cal1, cal2, delta, n, tau) / dp
        w -= s
        if abs(s) < 1e-9:
            break
    return w.real / (2 * np.pi), -w.imag


def cp_eff(col):  # gamma R / (gamma - 1) from the mean state (sound-speed consistent)
    g = col[ES_RHO] * col[ES_C] ** 2 / col[ES_P]
    return g * (col[ES_P] / (col[ES_RHO] * col[ES_T])) / (g - 1.0)


# the same n-tau response, now on the reacting flame (a larger lag, since the equilibrium
# flame's intrinsic quasi-steady response must be overcome)
TAU_R = 6.0e-3
sol_r = rijke_reacting(N, TAU_R)
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    res_r = eigenmodes(sol_r.problem, sol_r.x, freq_band=(50.0, 200.0), growth_band=(-250.0, 250.0), isentropic=True)
for mm in res_r.summary():
    flag = "  <-- UNSTABLE" if mm["unstable"] else ""
    print(f"f = {mm['freq_hz']:7.2f} Hz    growth = {mm['growth_rate']:+8.2f} 1/s{flag}")

# quantitative check against the equilibrium-flame dispersion relation
est_r = sol_r.table()
means = (est_r[ES_RHO, 1], est_r[ES_C, 1], est_r[ES_U, 1], est_r[ES_P, 1],
         est_r[ES_RHO, 2], est_r[ES_C, 2], est_r[ES_U, 2], est_r[ES_P, 2])
delta = 0.5 * (cp_eff(est_r[:, 1]) + cp_eff(est_r[:, 2])) * (est_r[ES_T, 2] - est_r[ES_T, 1])
cal1, cal2 = reacting_caloric(sol_r.problem, sol_r.x, est_r, 1), reacting_caloric(sol_r.problem, sol_r.x, est_r, 2)
f_fns, g_fns = max(zip(res_r.freqs, res_r.growth_rates), key=lambda t: t[1])
f_an, g_an = reacting_mode(means, cal1, cal2, delta, N, TAU_R, 2 * np.pi * f_fns)
print(f"\nmost-unstable mode:  FNS       f = {f_fns:6.2f} Hz, growth = {g_fns:+7.2f} 1/s")
print(f"                     analytic  f = {f_an:6.2f} Hz, growth = {g_an:+7.2f} 1/s")

res_r.plot_spectrum(title=f"Reacting Rijke spectrum  (H2/air, n={N}, tau={TAU_R*1e3:.0f} ms)").show()

## Summary

* The dynamic **source face** $\mathbf S(\omega)$ -- a flame transfer function or a fluctuating mass injection -- drops into the same operator $\mathbf A(\omega)$ the scattering and stability drivers already use; no new solver.
* For the heat release it lands on the **downstream energy row** with the verified sign, making the operator active; `eigenmodes` then finds the self-excited modes.
* FNS reproduces the **analytical** Rijke dispersion relation in frequency and growth rate, including the stability boundary -- the implementation is quantitatively correct.
* The **reacting** equilibrium flame is driven by the identical FTF, with its mean heat release taken from the reacting solve.  Because the perturbation maps use the gas's actual per-edge caloric coupling, the reacting case matches its own equilibrium-flame dispersion relation in frequency **and** growth rate -- the reacting Rijke is quantitatively verified, not just demonstrated.